# Linear Cryptanalysis of Block Ciphers

This notebook is the executable companion to [Linear Cryptanalysis.md](Linear%20Cryptanalysis.md). It follows the same progression while importing the checked implementation from `matsui1.py`. Run all cells from the `Matsui` directory.

## 1. Masks and binary inner products

For binary vectors, $\alpha\cdot x$ is the parity of the bits selected by $\alpha$. Bit positions in this project are zero-based and MSB-first.

In [ ]:
from matsui1 import (
    COMPANION_SBOX, PRESENT_SBOX, TOY_SPN_PBOX, TOY_SPN_SBOX,
    SPN, approximation_bias, approximation_correlation, count_matches,
    demo_spn_partial_attack, dot, estimate_data_complexity,
    extract_round_key_nibbles, format_lat, inverse_sbox,
    linear_approximation_table, matsui1_details, matsui2, matsui2_spn,
    piling_up_bias, rank_key_guesses, walsh_coefficient,
)

assert dot(0b1011, 0b1001) == 0
assert dot(0b1101, 0b1100) == 0

## 2. Linear approximation table

This project stores centered match counts: $B=\#\text{matches}-2^{n-1}$. Walsh coefficient is $W=2B$, normalized correlation is $C=W/2^n$, and bias is $\epsilon=C/2$.

In [ ]:
lat = linear_approximation_table(PRESENT_SBOX, 4)
print(format_lat(lat))

In [ ]:
alpha, beta = 0x9, 0x1
print('matches:', count_matches(alpha, beta, PRESENT_SBOX, 4))
print('LAT entry:', lat[alpha][beta])
print('Walsh:', walsh_coefficient(alpha, beta, PRESENT_SBOX, 4))
print('bias:', approximation_bias(alpha, beta, PRESENT_SBOX, 4))
print('correlation:', approximation_correlation(alpha, beta, PRESENT_SBOX, 4))
assert (lat[alpha][beta], approximation_bias(alpha, beta, PRESENT_SBOX, 4)) == (4, 0.25)

## 3. Negative bias and piling up

A negative correlation is useful: it means the complementary affine relation is favoured. Under the independence assumption, correlations multiply. For biases, $\epsilon=2^{r-1}\prod_i\epsilon_i$.

In [ ]:
eps1 = approximation_bias(0x9, 0x1, PRESENT_SBOX, 4)
eps2 = approximation_bias(0x1, 0x5, PRESENT_SBOX, 4)
combined = piling_up_bias([eps1, eps2])
print(eps1, eps2, combined, 'probability =', 0.5 + combined)
assert combined == -0.125

## 4. Adding a key and Matsui's Algorithm 1

If $Y=S(X\oplus K)$, then an approximation from $\alpha$ to $\beta$ exposes the parity $\alpha\cdot K$. Algorithm 1 recovers one such parity. The sign of the keyless correlation determines whether the majority decision is complemented.

In [ ]:
key = 0xA
messages = list(range(16))  # all distinct 4-bit plaintexts
ciphertexts = [PRESENT_SBOX[m ^ key] for m in messages]
result = matsui1_details(messages, ciphertexts, 0x9, 0x1, correlation_sign=+1)
print(result)
assert result.key_parity == dot(key, 0x9)

## 5. Matsui's Algorithm 2

Algorithm 2 guesses outer-round subkey bits, partially decrypts, and ranks candidates by $|T_0-T_1|$. On a 4-bit full codebook, equivalent candidates can tie; the example demonstrates the statistic rather than claiming unique recovery.

In [ ]:
k0, k1, k2, k3 = 0x3, 0xA, 0x5, 0xC
messages = list(range(16))
ciphertexts = [
    COMPANION_SBOX[COMPANION_SBOX[COMPANION_SBOX[m ^ k0] ^ k1] ^ k2] ^ k3
    for m in messages
]
scores = matsui2(messages, ciphertexts, 0xD, 0xD, inverse_sbox(COMPANION_SBOX))
ranking = rank_key_guesses(scores)
print('top candidates:', [(f'{k:X}', scores[k]) for k in ranking[:6]])
print('true final key rank:', ranking.index(k3) + 1)

## 6. Three-round trail in the 16-bit SPN

The trail `0x0B00 -> 0x0505` contains four active S-box approximations. Its component correlations are $1/2,-1/2,-1/2,-1/2$, so its correlation is $-1/16$ and bias is $-1/32$. The heuristic scale $1/\epsilon^2$ is 1024, but ranking 256 candidates reliably needs a larger experimental constant.

In [ ]:
trail_bias = piling_up_bias([0.25, -0.25, -0.25, -0.25])
print('trail bias:', trail_bias)
print('heuristic data scale:', estimate_data_complexity(trail_bias))
assert trail_bias == -1/32

## 7. Partial last-round subkey recovery

The endpoint mask activates the second and fourth final-round S-boxes. We therefore guess the corresponding whitening-key nibbles, packed as one 8-bit candidate. This recovers selected round-key bits, not the complete master key.

In [ ]:
best, actual, rank = demo_spn_partial_attack(8_192)
print(f'best={best:02X}, actual={actual:02X}, true rank={rank}')
assert best == actual == 0x6F and rank == 1

## 8. Validation and next steps

Run `python -m unittest -v test.py` for exact LAT checks, mask-propagation identities, encryption/decryption round trips, sign handling, and the deterministic partial attack. Continue with linear hulls, multiple and multidimensional approximations, success-probability models, and zero-correlation linear cryptanalysis using the references in `references.bib`.